# MIMO über die CLI - Test mit zwei Tönen

**Kann `usrp-client` MIMO?** Ja. Die CLI sendet die `.f32` roh - den MIMO-Modus trägst du im **Datei-Header**, nicht als Flag. Der Server erkennt MIMO am Magic `MIMO\x00\x00\x00\x00` am Dateianfang.

Dieses Notebook erzeugt eine MIMO-`.f32` mit **100 kHz** (Kanal 0) und **50 kHz** (Kanal 1), sendet sie per CLI und wertet das Ergebnis aus.

In [ ]:
import numpy as np, struct, matplotlib.pyplot as plt
from usrp_benchmark import USRPClient

SERVER = "129.132.24.210:80"          # fuer die CLI (-s)
HOST, PORT = "129.132.24.210", 80     # fuer USRPClient
TOKEN  = "rsahleanu-pbmmyfz5ggscdalo4rktbpkl"

USRPClient.setup(host=HOST, port=PORT, token=TOKEN)
info = USRPClient.info()
fs = int(info["sample_rate_hz"])
begin_guard = float(info.get("begin_guard_min_sec", 0.1))
print(f"fs = {fs/1e6:.3f} MHz | begin guard = {begin_guard*1000:.0f} ms | MIMO enabled: {info.get('mimo_enabled')}")

## Wie eine MIMO-`.f32` aussieht

```
Offset  Groesse  Inhalt
  0       8      Magic  b"MIMO\x00\x00\x00\x00"
  8       4      n_channels  (uint32, little-endian)
 12       4      n_samples   (uint32, pro Kanal)
 16       ...    ch0: I0 Q0 I1 Q1 ...  (n_samples Paare, float32)
               ch1: I0 Q0 I1 Q1 ...
```

Wichtig: **kanal-sequentiell** - erst alle Samples von ch0, dann ch1, nicht ueber die Kanaele verschachtelt. Eine SISO-`.f32` ist einfach `I0 Q0 I1 Q1 ...` **ohne** Header.

In [ ]:
def write_mimo_f32(path, sig2d):
    """sig2d: (n_samples, n_channels) komplex -> MIMO-.f32 mit Header."""
    arr = np.asarray(sig2d, dtype=np.complex64)
    n_samples, n_channels = arr.shape
    header = b"MIMO\x00\x00\x00\x00" + struct.pack("<II", n_channels, n_samples)
    out = np.empty(n_channels * n_samples * 2, dtype=np.float32)
    for ch in range(n_channels):
        base = ch * n_samples * 2
        out[base+0::2][:n_samples] = arr[:, ch].real
        out[base+1::2][:n_samples] = arr[:, ch].imag
    open(path, "wb").write(header + out.tobytes())
    return header

N = 100_000
t = np.arange(N) / fs
ch0 = 0.5 * np.exp(2j*np.pi*100e3*t)     # 100 kHz auf Kanal 0
ch1 = 0.5 * np.exp(2j*np.pi* 50e3*t)     #  50 kHz auf Kanal 1
tx = np.stack([ch0, ch1], axis=1)        # Shape (N, 2)

header = write_mimo_f32("mimo.f32", tx)
print("Header (hex):", header.hex())
print("magic     :", header[:8])
print("channels  :", struct.unpack("<I", header[8:12])[0])
print("samples/ch:", struct.unpack("<I", header[12:16])[0])

## Senden ueber die CLI

Die naechste Zelle ruft `usrp-client` direkt auf (`!` = Shell-Befehl). Im Terminal waere es:

```bash
usrp-client -i mimo.f32 -o out.f32 -s 129.132.24.210:80 -t DEIN-TOKEN
```

Gibt der Server `mimo_disabled` zurueck, ist MIMO am Server aus (`MIMO_ENABLED`) oder es sind weniger als 2 Kanaele in der Hardware-Inventory.

In [ ]:
!usrp-client -i mimo.f32 -o out.f32 -s {SERVER} -t {TOKEN}

## Ergebnis auswerten

`out.f32` ist ebenfalls ein MIMO-Blob. Wir lesen ihn, schneiden pro Kanal das Nutzsignal hinter dem Begin-Guard heraus und suchen den Spektral-Peak. Erwartet: Kanal 0 bei ~100 kHz, Kanal 1 bei ~50 kHz.

In [ ]:
def read_mimo_f32(path):
    data = open(path, "rb").read()
    assert data[:8] == b"MIMO\x00\x00\x00\x00", "keine MIMO-Datei (Header fehlt)"
    n_channels, n_samples = struct.unpack("<II", data[8:16])
    raw = np.frombuffer(data[16:16+n_channels*n_samples*8], dtype=np.float32)
    out = np.empty((n_samples, n_channels), np.complex64)
    for ch in range(n_channels):
        b = ch*n_samples*2
        out[:, ch] = raw[b:b+n_samples*2:2] + 1j*raw[b+1:b+n_samples*2:2]
    return out

rx = read_mimo_f32("out.f32")
print("RX shape:", rx.shape, "(samples, channels)")

guard = int(begin_guard * fs)
expected_khz = [100, 50]
n_ch = rx.shape[1]
fig, axes = plt.subplots(1, n_ch, figsize=(13, 4), squeeze=False)
for ch in range(n_ch):
    core = rx[guard:guard+N, ch]
    X = np.fft.fftshift(np.fft.fft(core))
    f = np.fft.fftshift(np.fft.fftfreq(len(core), d=1/fs)) / 1e3   # kHz
    peak = f[int(np.argmax(np.abs(X)))]
    ax = axes[0][ch]
    ax.plot(f, 20*np.log10(np.abs(X) + 1e-9), lw=0.8)
    ax.axvline(expected_khz[ch], color="r", ls="--", lw=0.8)
    ax.set_title(f"Kanal {ch}: Peak {peak:.1f} kHz (erwartet {expected_khz[ch]} kHz)")
    ax.set_xlabel("Frequenz [kHz]"); ax.set_ylabel("|X(f)| [dB]")
    ax.set_xlim(-fs/2/1e3, fs/2/1e3); ax.grid(True)
plt.tight_layout(); plt.show()